Tansform Circuits Data

- Read bronze circuits table
- Keep only the columns required for analytics.
- standardise column names using snake_case.
- Rename columns to make them more meaningful.
- Filer out rows where circuit_id is null
- Remobe duplicate records.
- Transform values of columns circuit_name and locality to Title case
- write the trandformed data to silver circuits table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
bronze_table = f"{catalog_nameone}.{bronze_schema}.circuits"
silver_table = f"{catalog_nameone}.{silver_schema}.circuits"

**2. Keep only the columns required for analytics.**

In [0]:
circuits_df = spark.read.table(bronze_table)

**2. Keep only the columns required for analytics.**

In [0]:
from pyspark.sql.functions import col

circuits_selected_df = circuits_df.select(
col("circuitId"),
col("circuitName"),
col("lat"),
col("long"),
col("locality"),
col("country"),
col("ingestion_timestamp"),
col("source_file")
)

**3. standardise column names using snake_case.**

In [0]:
circuits_renamed_df = (
  circuits_selected_df
    .withColumnsRenamed({
      "circuitId": "circuit_id",
      "circuitName": "circuit_name",
      "lat": "latitude", 
      "long": "longitude"
    })
)

**5. Filer out rows where circuit_id is null**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

circuit_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

**6. Remove duplicate records.**

In [0]:
circuits_distinct_df = circuit_valid_df.dropDuplicates(["circuit_id"])


%md
**7. Transform values of columns circuit_name and locality to Title case**

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality"))) 
    
)

**8. write the trandformed data to silver circuits table**

In [0]:
circuits_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))